# Model Evaluation

Evaluate model performance and inspect prediction quality.

In [33]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

In [34]:
df = pd.read_csv("../data/processed/engineered_air_quality.csv")

df["Date"] = pd.to_datetime(df["Date"])

In [35]:
train_df = df[df["Date"] < "2019-01-01"].copy()

val_df = df[
    (df["Date"] >= "2019-01-01") &
    (df["Date"] < "2020-01-01")
].copy()

test_df = df[df["Date"] >= "2020-01-01"].copy()

In [36]:
num_features = [
    "Valid_AQI",
    "PM2.5",
    "PM10",
    "NO2",
    "CO",
    "SO2",
    "O3",

    "AQI_Lag1",
    "AQI_Lag2",
    "AQI_Lag3",
    "AQI_Lag7",
    "AQI_Roll3",
    "AQI_Roll7",

    "PM2.5_Lag1",
    "PM10_Lag1",
    "NO2_Lag1",
    "CO_Lag1",
    "SO2_Lag1",
    "O3_Lag1",

    "AQI_Change1",
    "PM25_Change1",

    "Month_Sin",
    "Month_Cos",
    "DayOfYear_Sin",
    "DayOfYear_Cos"
]

cat_features = [
    "City",
    "Season",
    "DayOfWeek"
]

features = num_features + cat_features

In [37]:
X_train = train_df[features]
y_train = train_df["Target_AQI"]

X_val = val_df[features]
y_val = val_df["Target_AQI"]

X_test = test_df[features]
y_test = test_df["Target_AQI"]

In [38]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_features),
    ("cat", categorical_transformer, cat_features)
])

In [30]:
from xgboost import XGBRegressor

configs = [
    {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "max_depth": 4
    },
    {
        "n_estimators": 500,
        "learning_rate": 0.03,
        "max_depth": 6
    },
    {
        "n_estimators": 700,
        "learning_rate": 0.02,
        "max_depth": 6
    },
    {
        "n_estimators": 500,
        "learning_rate": 0.03,
        "max_depth": 4
    }
]

In [39]:
tuning_results = []

for cfg in configs:
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(
            n_estimators=cfg["n_estimators"],
            learning_rate=cfg["learning_rate"],
            max_depth=cfg["max_depth"],
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        ))
    ])

    model.fit(X_train, y_train)

    pred = model.predict(X_val)

    mae_score = mean_absolute_error(y_val, pred)

    rmse_score = np.sqrt(
        mean_squared_error(y_val, pred)
    )

    r2_score_val = r2_score(y_val, pred)

    tuning_results.append({
        **cfg,
        "MAE": mae_score,
        "RMSE": rmse_score,
        "R2": r2_score_val
    })

In [40]:
tuning_df = pd.DataFrame(tuning_results)

tuning_df.sort_values("MAE")

,n_estimators,learning_rate,max_depth,MAE,RMSE,R2
2,700,0.02,6,17.184849,26.098227,0.906661
1,500,0.03,6,17.224760,26.102898,0.906628
0,300,0.05,4,17.386380,26.371234,0.904698
3,500,0.03,4,17.397102,26.408761,0.904427


In [41]:
final_train = pd.concat(
    [train_df, val_df],
    ignore_index=True
)

In [42]:
X_final_train = final_train[features]
y_final_train = final_train["Target_AQI"]

In [43]:
final_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=500,
        learning_rate=0.03,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    ))
])

In [44]:
final_model.fit(
    X_final_train,
    y_final_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](28,)","['Valid_AQI','PM2.5','PM10',...,'City','Season','DayOfWeek']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,28
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concaten

In [45]:
final_pred = final_model.predict(X_test)

In [46]:
test_mae = mean_absolute_error(
    y_test,
    final_pred
)

test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        final_pred
    )
)

test_r2 = r2_score(
    y_test,
    final_pred
)

print("Final Test Performance")
print("MAE:", round(test_mae, 2))
print("RMSE:", round(test_rmse, 2))
print("R2:", round(test_r2, 3))

Final Test Performance
MAE: 14.32
RMSE: 23.03
R2: 0.868


In [47]:
test_baseline = test_df[
    ["Valid_AQI", "Target_AQI"]
].dropna()

baseline_test_pred = test_baseline["Valid_AQI"]

baseline_test_mae = mean_absolute_error(
    test_baseline["Target_AQI"],
    baseline_test_pred
)

baseline_test_rmse = np.sqrt(
    mean_squared_error(
        test_baseline["Target_AQI"],
        baseline_test_pred
    )
)

baseline_test_r2 = r2_score(
    test_baseline["Target_AQI"],
    baseline_test_pred
)

print("Test Baseline")
print("MAE:", round(baseline_test_mae, 2))
print("RMSE:", round(baseline_test_rmse, 2))
print("R2:", round(baseline_test_r2, 3))

Test Baseline
MAE: 17.25
RMSE: 29.36
R2: 0.785


In [ ]:
import plotly.express as px

evaluation = test_df[
    ["City", "Date", "Target_AQI"]
].copy()

evaluation["Predicted_AQI"] = final_pred # type: ignore

In [49]:
fig = px.scatter(
    evaluation,
    x="Target_AQI",
    y="Predicted_AQI",
    title="Actual vs Predicted AQI",
    labels={
        "Target_AQI": "Actual AQI",
        "Predicted_AQI": "Predicted AQI"
    }
)

fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=500,
    y1=500,
    line=dict(dash="dash")
)

fig.show()

In [50]:
delhi_eval = evaluation[
    evaluation["City"] == "Delhi"
]

fig = px.line(
    delhi_eval,
    x="Date",
    y=["Target_AQI", "Predicted_AQI"],
    title="Delhi: Actual vs Predicted AQI"
)

fig.show()

In [51]:
evaluation["Residual"] = (
    evaluation["Target_AQI"] -
    evaluation["Predicted_AQI"]
)

In [52]:
fig = px.histogram(
    evaluation,
    x="Residual",
    nbins=50,
    title="Prediction Error Distribution"
)

fig.show()

In [53]:
city_results = []

for city, g in evaluation.groupby("City"):
    city_results.append({
        "City": city,
        "MAE": mean_absolute_error(
            g["Target_AQI"],
            g["Predicted_AQI"]
        ),
        "RMSE": np.sqrt(
            mean_squared_error(
                g["Target_AQI"],
                g["Predicted_AQI"]
            )
        )
    })

city_results = pd.DataFrame(city_results)

city_results.sort_values("MAE")

,City,MAE,RMSE
2,Bengaluru,6.389202,8.220259
6,Hyderabad,7.333989,9.578276
10,Thiruvananthapuram,7.676170,16.189533
0,Amaravati,9.389829,12.521046
7,Jaipur,10.744508,14.843224
3,Chennai,11.120879,18.404326
11,Visakhapatnam,14.678015,20.594863
8,Kolkata,16.727359,23.500860
1,Amritsar,18.731854,37.818324
4,Delhi,21.582005,28.558416


In [54]:
import joblib

In [55]:
joblib.dump(
    final_model,
    "../models/aqi_forecasting_model.pkl"
)

['../models/aqi_forecasting_model.pkl']